In [ ]:
# Requirements: pandas, numpy, scikit-learn, torch (PyTorch)
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Tuple, List, Dict

# ---- metrics ----
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# ---- deep learning ----
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# --------------------------
# Utility: determinism
# --------------------------
def set_seed(seed: int):
    import random, os
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# --------------------------
# Fairness helpers (binary s)
# --------------------------
def SPD_gap(y_hat: np.ndarray, s: np.ndarray) -> float:
    # Statistical parity difference |P(y=1|s=1) - P(y=1|s=0)|
    p1 = (y_hat[s==1]==1).mean() if (s==1).any() else 0.0
    p0 = (y_hat[s==0]==1).mean() if (s==0).any() else 0.0
    return abs(p1 - p0)

def EOD_gap(y_true: np.ndarray, y_hat: np.ndarray, s: np.ndarray) -> float:
    # Equalized odds difference on TPR: |TPR(s=1) - TPR(s=0)|
    pos1 = (s==1) & (y_true==1)
    pos0 = (s==0) & (y_true==1)
    tpr1 = (y_hat[pos1]==1).mean() if pos1.any() else 0.0
    tpr0 = (y_hat[pos0]==1).mean() if pos0.any() else 0.0
    return abs(tpr1 - tpr0)

# --------------------------
# Model
# --------------------------
class MLP(nn.Module):
    def __init__(self, in_dim: int, p_drop=0.20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(128, 1)  # single logit
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)  # logits

# --------------------------
# Training / evaluation
# --------------------------
@dataclass
class TrainConfig:
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 100
    batch_size: int = 256
    patience: int = 10  # early stopping on val AUROC

def train_one_fold(X_tr, y_tr, X_val, y_val, cfg: TrainConfig, seed: int, device='cpu'):
    set_seed(seed)
    model = MLP(in_dim=X_tr.shape[1], p_drop=0.20).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3, verbose=False)

    ds_tr = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).float())
    ds_val = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).float())
    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=cfg.batch_size, shuffle=False, drop_last=False)

    bcelogits = nn.BCEWithLogitsLoss()
    best_auroc, best_state, no_improve = -np.inf, None, 0

    for ep in range(cfg.epochs):
        # train
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = bcelogits(logits, yb)
            loss.backward()
            opt.step()

        # validate
        model.eval()
        with torch.no_grad():
            all_logits, all_y = [], []
            for xb, yb in dl_val:
                xb = xb.to(device)
                logits = model(xb)
                all_logits.append(logits.detach().cpu().numpy())
                all_y.append(yb.numpy())
            val_logits = np.concatenate(all_logits)
            val_y = np.concatenate(all_y)
            val_probs = 1/(1+np.exp(-val_logits))
            try:
                val_auroc = roc_auc_score(val_y, val_probs)
            except Exception:
                val_auroc = 0.5

        # scheduler + early stopping
        scheduler.step(val_auroc)
        if val_auroc > best_auroc + 1e-6:
            best_auroc = val_auroc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= cfg.patience:
                break

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def pick_threshold_for_f1(y_val: np.ndarray, p_val: np.ndarray) -> float:
    # search thresholds on sorted unique probs (fast & deterministic)
    thr_candidates = np.unique(p_val)
    if len(thr_candidates) > 1024:
        thr_candidates = np.quantile(p_val, np.linspace(0,1,1025))  # cap search
    best_thr, best_f1 = 0.5, -1
    for thr in thr_candidates:
        y_hat = (p_val >= thr).astype(int)
        f1 = f1_score(y_val, y_hat, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return float(best_thr)

def infer_probs(model: nn.Module, X: np.ndarray, batch_size=4096, device='cpu') -> np.ndarray:
    model.eval()
    ds = TensorDataset(torch.from_numpy(X).float(), torch.zeros(len(X)))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    out = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            logits = model(xb).detach().cpu().numpy()
            out.append(logits)
    logits = np.concatenate(out)
    return 1/(1+np.exp(-logits))

# --------------------------
# Cross-validation scaffold
# --------------------------
def mean_ci95(x: List[float]) -> Tuple[float, float]:
    # mean ± 95% CI (Student-t if SciPy available; else normal 1.96)
    import math
    n = len(x)
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1)) if n > 1 else 0.0
    sem = sd / math.sqrt(n) if n > 0 else 0.0
    try:
        from scipy.stats import t
        tval = t.ppf(0.975, df=max(n-1,1))
    except Exception:
        tval = 1.96
    return mean, tval*sem

def run_cv_mlp(X: pd.DataFrame, y: np.ndarray, s: np.ndarray, device='cpu'):
    assert set(np.unique(y)).issubset({0,1}), "y must be binary {0,1}"
    assert set(np.unique(s)).issubset({0,1}), "s must be binary {0,1}"

    # Column partition
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    pre = ColumnTransformer([
        ('num', StandardScaler(with_mean=True, with_std=True), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ], remainder='drop')
    # Note: For scikit-learn <1.2 use sparse=False instead of sparse_output=False.

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=4765416)

    accs, f1s, aucs, spds, eods = [], [], [], [], []
    cfg = TrainConfig()

    for fold_id, (tr_idx, te_idx) in enumerate(skf.split(X, y)):
        seed = 4765416 + fold_id

        # Fit preprocessing on train only
        X_tr_full, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr_full, y_te = y[tr_idx], y[te_idx]
        s_tr_full, s_te = s[tr_idx], s[te_idx]

        X_tr_full_t = pre.fit_transform(X_tr_full)
        X_te_t = pre.transform(X_te)

        # Carve validation from training (stratified)
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        (tr2_idx, va_idx) = next(sss.split(X_tr_full_t, y_tr_full))
        X_tr, X_val = X_tr_full_t[tr2_idx], X_tr_full_t[va_idx]
        y_tr, y_val = y_tr_full[tr2_idx], y_tr_full[va_idx]
        s_tr, s_val = s_tr_full[tr2_idx], s_tr_full[va_idx]

        # Train
        model = train_one_fold(X_tr, y_tr, X_val, y_val, cfg, seed=seed, device=device)

        # Pick threshold on validation for best F1
        p_val = infer_probs(model, X_val, device=device)
        thr = pick_threshold_for_f1(y_val, p_val)

        # Evaluate on test
        p_te = infer_probs(model, X_te, device=device)
        y_hat = (p_te >= thr).astype(int)

        accs.append(accuracy_score(y_te, y_hat))
        f1s.append(f1_score(y_te, y_hat, zero_division=0))
        try:
            aucs.append(roc_auc_score(y_te, p_te))
        except Exception:
            aucs.append(0.5)
        spds.append(SPD_gap(y_hat, s_te))
        eods.append(EOD_gap(y_te, y_hat, s_te))

    # Aggregate (mean ± 95% CI)
    def fmt(m, h): return f"{m:.3f} ± {h:.3f}"

    acc_m, acc_h = mean_ci95(accs)
    f1_m, f1_h   = mean_ci95(f1s)
    auc_m, auc_h = mean_ci95(aucs)
    spd_m, spd_h = mean_ci95(spds)
    eod_m, eod_h = mean_ci95(eods)

    print("MLP (DL baseline) — mean ± 95% CI (decimals)")
    print("Accuracy:", fmt(acc_m, acc_h))
    print("F1-score:", fmt(f1_m, f1_h))
    print("AUROC   :", fmt(auc_m, auc_h))
    print("SPD |Δ| :", fmt(spd_m, spd_h))
    print("EOD |Δ| :", fmt(eod_m, eod_h))

    # Optional: percentages
    print("\n(Percentages)")
    print("Accuracy (%):", fmt(100*acc_m, 100*acc_h))
    print("F1-score (%):", fmt(100*f1_m, 100*f1_h))
    print("AUROC   (%):", fmt(100*auc_m, 100*auc_h))
    print("SPD |Δ| (%):", fmt(100*spd_m, 100*spd_h))
    print("EOD |Δ| (%):", fmt(100*eod_m, 100*eod_h))


# --------------------------
# Example usage:
# X: pandas DataFrame of features
# y: numpy array of {0,1}
# s: numpy array of {0,1} (sensitive attribute)
# --------------------------
# X, y, s = ""
# run_cv_mlp(X, y, s, device='cuda' if torch.cuda.is_available() else 'cpu')
